# Andre's Flat Relational Tables — Scaleup

This notebook reorganises the scaleup attribution data from its nested `sentence_detail` structure into
four flat, database-style tables, as requested by the thesis tutor (Andre). The objective is that every
reported metric — the Attributable-to-Identified-Sources rate (AIS) and the Position-Adjusted Word
Coverage (PAWC) — can be reconstructed from simple aggregations over a single atomic table (**Table 4**),
without any consumer having to parse nested arrays.

The notebook is **read-only** with respect to all existing artefacts: it loads the gold and silver inputs,
derives the four tables in memory, and writes only the four new flat tables to the gold layer.

The scaleup design uses **k = 8 repeated runs** per (engine, query) cell; the *k = 3* figure the tutor
recalled refers to the earlier pilot, not this scaleup. The analysis is reported in a formal,
third-person register with British spelling.

**Tables produced:** Table 1 (responses), Table 2 (sentences), Table 3 (citations), and Table 4
(entailment scores — the atomic deliverable).

## Section 0 — Setup and data loading

Imports the libraries, loads the three primary inputs, and defines the `is_artifact` classifier (the same
instrument of record used throughout the attribution pipeline) together with the `response_id`
constructor. A unique `response_id` of the form `{engine}__{query_id}__r{run_index}` — matching the bronze
filename convention — is attached to a working copy of the cleaned frame.

Two attributes requested in the table schemas, `model_served` (Table 1) and `fetch_status` (Table 3), are
not carried in the three primary inputs; they are read, read-only, from the silver `responses` and
`sources` tables respectively. No input file is modified.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from IPython.display import display

# resolve the project root so relative data paths work irrespective of launch directory
_guard = 0
while not os.path.isdir('data/scaleup/gold') and _guard < 6:
    os.chdir('..'); _guard += 1
print('working directory:', os.getcwd())

# --- input files ---
V2      = 'data/scaleup/silver/nli_scaleupv2.parquet'            # has source_scores (continuous)
CLEANED = 'data/scaleup/silver/nli_scaleup_cleaned.parquet'      # has cleaned_* columns
CITES   = 'data/scaleup/silver/citations_scaleup.parquet'      # cited URLs per response
RESP    = 'data/scaleup/silver/responses_scaleup.parquet'      # source of model_served (read-only)
SOURCES = 'data/scaleup/silver/sources_scaleup.parquet'        # source of fetch_status (read-only)

INPUT_FILES = [V2, CLEANED, CITES, RESP, SOURCES]
# fingerprint the inputs so the final cell can confirm none were modified
_FP = {p: (os.path.getsize(p), round(os.path.getmtime(p), 3)) for p in INPUT_FILES}

v2      = pd.read_parquet(V2)
cleaned = pd.read_parquet(CLEANED)
cites   = pd.read_parquet(CITES)

print(f'nli_scaleupv2        {v2.shape}')
print(f'nli_scaleup_cleaned  {cleaned.shape}')
print(f'citations_scaleup    {cites.shape}')


def is_artifact(text: str) -> bool:
    t = text.strip()
    if not t:
        return True
    if re.match(r'^https?://\S+$', t):
        return True
    if t.startswith('#'):
        return True
    if re.match(r'^[-*=_]{3,}\s*$', t):
        return True
    if re.match(r'^(sources?|references?|citations?)\s*:?\s*$', t, re.IGNORECASE):
        return True
    if len(t.split()) <= 3:
        return True
    if re.match(r'^[\#\-\*\d\.\s]{1,10}$', t):
        return True
    return False


def _L(x):
    # struct/list parquet columns come back from pyarrow as numpy arrays
    return x.tolist() if isinstance(x, np.ndarray) else x

def make_response_id(df):
    # {engine}__{query_id}__r{run_index}  (matches the bronze filename convention)
    return df['engine'] + '__' + df['query_id'] + '__r' + df['run_index'].astype(str)


# working copy of the cleaned frame carrying a unique response_id
work = cleaned.copy()
work['response_id'] = make_response_id(work)
print('\nunique response_id:', work['response_id'].nunique(), 'of', len(work), 'rows')

working directory: /Users/ganenthraravindran/Desktop/Thesis Data Pilot


nli_scaleupv2        (9992, 15)
nli_scaleup_cleaned  (9992, 20)
citations_scaleup    (58851, 9)

unique response_id: 9992 of 9992 rows


## Section 1 — Table 1: Responses

One row per response: a projection of the cleaned frame augmented with `response_id` and `model_served`.

**Validation:** exactly 9,992 rows; `response_id` unique; all specified columns present.

In [2]:
# model_served is not in the gold cleaned frame; it is taken read-only from the silver responses table
resp_meta = (pd.read_parquet(RESP, columns=['engine', 'query_id', 'run_index', 'model_served'])
               .drop_duplicates(['engine', 'query_id', 'run_index']))

table1 = work[['response_id', 'query_id', 'engine', 'run_index',
               'n_sentences', 'n_artifacts', 'cleaned_n', 'n_sources_cited',
               'n_sources_fetched_ok', 'cleaned_ais', 'cleaned_pawc_total', 'pawc_norm']].copy()
table1 = table1.merge(resp_meta, on=['engine', 'query_id', 'run_index'], how='left')

# place model_served immediately after run_index, as specified
table1 = table1[['response_id', 'query_id', 'engine', 'run_index', 'model_served',
                 'n_sentences', 'n_artifacts', 'cleaned_n', 'n_sources_cited',
                 'n_sources_fetched_ok', 'cleaned_ais', 'cleaned_pawc_total', 'pawc_norm']]

print('Table 1 - Responses:', table1.shape)
display(table1.head(5))

Table 1 - Responses: (9992, 13)


,response_id,query_id,engine,run_index,model_served,n_sentences,n_artifacts,cleaned_n,n_sources_cited,n_sources_fetched_ok,cleaned_ais,cleaned_pawc_total,pawc_norm
0,chatgpt__gb_00372fa592bfcd6b__r1,gb_00372fa592bfcd6b,chatgpt,1,gpt-5.5-2026-04-23,2,0,2,0,0,0.0,0.0,NaN
1,chatgpt__gb_008096c51bc414ba__r1,gb_008096c51bc414ba,chatgpt,1,gpt-5.5-2026-04-23,3,1,2,0,0,0.0,0.0,NaN
2,chatgpt__gb_021e53327e1ee5a9__r1,gb_021e53327e1ee5a9,chatgpt,1,gpt-5.5-2026-04-23,18,5,13,0,0,0.0,0.0,NaN
3,chatgpt__gb_05118b5a6fcf8e43__r1,gb_05118b5a6fcf8e43,chatgpt,1,gpt-5.5-2026-04-23,17,1,16,0,0,0.0,0.0,NaN
4,chatgpt__gb_0515f875dced5ac8__r1,gb_0515f875dced5ac8,chatgpt,1,gpt-5.5-2026-04-23,2,0,2,0,0,0.0,0.0,NaN


In [3]:
# --- Table 1 validation ---
expected_cols = ['response_id', 'query_id', 'engine', 'run_index', 'model_served', 'n_sentences',
                 'n_artifacts', 'cleaned_n', 'n_sources_cited', 'n_sources_fetched_ok',
                 'cleaned_ais', 'cleaned_pawc_total', 'pawc_norm']
print('rows == 9992                :', len(table1) == 9992, f'({len(table1)})')
print('response_id unique          :', bool(table1['response_id'].is_unique),
      f"({int(table1['response_id'].duplicated().sum())} duplicates)")
print('all columns present & ordered:', list(table1.columns) == expected_cols)

rows == 9992                : True (9992)
response_id unique          : True (0 duplicates)
all columns present & ordered: True


## Section 2 — Table 2: Sentences

The `sentence_detail` array is exploded to one row per sentence per response. `sentence_idx` is the
stored 1-based `idx`; `position_weight` is the stored `w`; `is_supported` is the stored binary flag;
`is_artifact` is recomputed by the classifier; `n_supporting_sources` is the length of the stored
`sources` list (the positions reaching support at the operating threshold).

**Validation:** total rows equal the sum of `n_sentences`; per-engine artefact rates match the known
values (Kimi ~46%, ChatGPT ~20%, Claude ~16%, Gemini ~14%, Perplexity ~4%); every `response_id` is in
Table 1.

In [4]:
rows = []
for r in work.itertuples(index=False):
    rid = r.response_id
    for e in _L(r.sentence_detail):
        text = str(e.get('text', ''))
        rows.append((rid, int(e['idx']), text, int(e.get('wc', 0)), float(e.get('w', 0.0)),
                     bool(e.get('supported', False)), is_artifact(text),
                     len(_L(e.get('sources')) or [])))

table2 = pd.DataFrame(rows, columns=['response_id', 'sentence_idx', 'sentence_text', 'word_count',
                                     'position_weight', 'is_supported', 'is_artifact',
                                     'n_supporting_sources'])
print('Table 2 - Sentences:', table2.shape)
display(table2.head(10))

print('\nper-engine sentence counts:')
print(table2.merge(table1[['response_id', 'engine']], on='response_id').groupby('engine').size())

Table 2 - Sentences: (219897, 8)


,response_id,sentence_idx,sentence_text,word_count,position_weight,is_supported,is_artifact,n_supporting_sources
0,chatgpt__gb_00372fa592bfcd6b__r1,1,**BMI (Body Mass Index) standard categories fo...,95,1.0000,False,False,0
1,chatgpt__gb_00372fa592bfcd6b__r1,2,"It may be less accurate for athletes, older ad...",20,0.5000,False,False,0
2,chatgpt__gb_008096c51bc414ba__r1,1,Here are some **field trip ideas** organized b...,9,1.0000,False,False,0
3,chatgpt__gb_008096c51bc414ba__r1,2,"If you tell me the **grade level, location, bu...",16,0.6667,False,False,0
4,chatgpt__gb_008096c51bc414ba__r1,3,## Science & Nature\n- **Science museum**\n- *...,340,0.3333,False,True,0
5,chatgpt__gb_021e53327e1ee5a9__r1,1,Nitrogen is used in plastic welding mainly bec...,20,1.0000,False,False,0
6,chatgpt__gb_021e53327e1ee5a9__r1,2,Key reasons:\n\n1.,3,0.9444,False,True,0
7,chatgpt__gb_021e53327e1ee5a9__r1,3,**Prevents oxidation**\n - When plastic is h...,17,0.8889,False,False,0
8,chatgpt__gb_021e53327e1ee5a9__r1,4,"- This can cause discoloration, brittleness, w...",11,0.8333,False,False,0
9,chatgpt__gb_021e53327e1ee5a9__r1,5,"- Nitrogen is mostly inert, so it reduces oxid...",11,0.7778,False,False,0



per-engine sentence counts:
engine
chatgpt       39048
claude        56350
gemini        61264
kimi          58654
perplexity     4581
dtype: int64


In [5]:
# --- Table 2 validation ---
expected_rows = int(work['n_sentences'].sum())
print('rows == sum(n_sentences)   :', len(table2) == expected_rows, f'({len(table2)} vs {expected_rows})')

t2e = table2.merge(table1[['response_id', 'engine']], on='response_id')
print('\nartefact rate per engine (%)  [expect Kimi~46, ChatGPT~20, Claude~16, Gemini~14, Perplexity~4]:')
print(t2e.groupby('engine')['is_artifact'].mean().mul(100).round(1))

print('\nevery response_id in Table 1:', bool(table2['response_id'].isin(set(table1['response_id'])).all()))

rows == sum(n_sentences)   : True (219897 vs 219897)

artefact rate per engine (%)  [expect Kimi~46, ChatGPT~20, Claude~16, Gemini~14, Perplexity~4]:
engine
chatgpt       19.8
claude        15.7
gemini        14.2
kimi          46.1
perplexity     3.7
Name: is_artifact, dtype: float64

every response_id in Table 1: True


## Section 3 — Table 3: Citations

One row per (response, cited source), built by joining the cleaned `response_id` to the citations table
on (engine, query_id, run_index). `fetch_status` is taken read-only from the silver sources table (only
the `url_canonical` and `fetch_status` columns are read — never the large `cleaned_text` body).

**Validation:** total rows equal the citations table (58,851); every `response_id` is in Table 1; no
duplicate (response_id, citation_position) pairs.

In [6]:
# fetch_status joined from the silver sources table (small two-column read; cleaned_text never loaded)
fetch = (pd.read_parquet(SOURCES, columns=['url_canonical', 'fetch_status'])
           .drop_duplicates('url_canonical'))

key = table1[['response_id', 'engine', 'query_id', 'run_index']]
table3 = (cites.merge(key, on=['engine', 'query_id', 'run_index'], how='inner')
               .merge(fetch, on='url_canonical', how='left')
               .rename(columns={'position': 'citation_position'}))
table3 = table3[['response_id', 'citation_position', 'url_canonical', 'domain', 'fetch_status']]

print('Table 3 - Citations:', table3.shape)
display(table3.head(10))

print('\nper-engine citation counts:')
print(table3.merge(table1[['response_id', 'engine']], on='response_id').groupby('engine').size())

Table 3 - Citations: (58851, 5)


,response_id,citation_position,url_canonical,domain,fetch_status
0,chatgpt__gb_05118b5a6fcf8e43__r5,1,https://usaartnews.com/news/artistic-collabora...,usaartnews.com,ok
1,chatgpt__gb_05118b5a6fcf8e43__r5,2,https://www.harpersbazaar.com/fashion/fashion-...,harpersbazaar.com,ok
2,chatgpt__gb_05118b5a6fcf8e43__r5,3,https://www.marieclaire.co.uk/fashion/7-iconic...,marieclaire.co.uk,ok
3,chatgpt__gb_05118b5a6fcf8e43__r5,4,https://www.guggenheim.org/wp-content/uploads/...,guggenheim.org,ok
4,chatgpt__gb_05118b5a6fcf8e43__r5,5,https://press.moma.org/wp-content/uploads/2023...,press.moma.org,ok
5,chatgpt__gb_05118b5a6fcf8e43__r5,6,https://press.moma.org/wp-content/uploads/2023...,press.moma.org,ok
6,chatgpt__gb_05118b5a6fcf8e43__r5,7,https://en.wikipedia.org/wiki/Beyond_Granite%3...,en.wikipedia.org,http_403
7,chatgpt__gb_05118b5a6fcf8e43__r5,8,https://www.icp.org/news/the-international-cen...,icp.org,ok
8,chatgpt__gb_05118b5a6fcf8e43__r5,9,https://www.harpersbazaar.com/beauty/makeup/a4...,harpersbazaar.com,ok
9,chatgpt__gb_05118b5a6fcf8e43__r5,10,https://en.wikipedia.org/wiki/Creativity_Explored,en.wikipedia.org,http_403



per-engine citation counts:
engine
chatgpt        1403
claude        10314
gemini        14263
kimi          18874
perplexity    13997
dtype: int64


In [7]:
# --- Table 3 validation ---
print('rows == citations rows      :', len(table3) == len(cites), f'({len(table3)} vs {len(cites)})')
print('every response_id in Table 1:', bool(table3['response_id'].isin(set(table1['response_id'])).all()))
dups = int(table3.duplicated(['response_id', 'citation_position']).sum())
print('duplicate (response_id, citation_position):', dups)

rows == citations rows      : True (58851 vs 58851)
every response_id in Table 1: True
duplicate (response_id, citation_position): 0


## Section 4 — Table 4: Entailment Scores (the key deliverable)

One row per (sentence, source) pair carrying the continuous DeBERTa entailment probability. This is the
atomic table from which AIS and PAWC are recomputed in Section 5. Sentences with no fetched sources
contribute no rows, which is correct.

**Validation:** total rows equal the sum of `nli_evaluations` (one per NLI evaluation); `entail_score`
lies in [0, 1]; every `response_id` is in Table 1; a spot check cross-references one response against the
raw nested structure.

In [8]:
rows = []
for r in v2.itertuples(index=False):
    rid = r.engine + '__' + r.query_id + '__r' + str(r.run_index)
    for e in _L(r.sentence_detail):
        idx = int(e['idx'])
        for s in (_L(e.get('source_scores')) or []):
            rows.append((rid, idx, int(s['pos']), float(s['score'])))

table4 = pd.DataFrame(rows, columns=['response_id', 'sentence_idx', 'citation_position', 'entail_score'])
print('Table 4 - Entailment scores:', table4.shape)
display(table4.head(10))

print('\nper-engine row counts:')
print(table4.merge(table1[['response_id', 'engine']], on='response_id').groupby('engine').size())

Table 4 - Entailment scores: (1181792, 4)


,response_id,sentence_idx,citation_position,entail_score
0,chatgpt__gb_05912ccf2a3c68f8__r1,1,2,0.03343
1,chatgpt__gb_05912ccf2a3c68f8__r1,2,2,0.03157
2,chatgpt__gb_05912ccf2a3c68f8__r1,3,2,0.53563
3,chatgpt__gb_05912ccf2a3c68f8__r1,4,2,0.45338
4,chatgpt__gb_05912ccf2a3c68f8__r1,5,2,0.35830
5,chatgpt__gb_05912ccf2a3c68f8__r1,6,2,0.78460
6,chatgpt__gb_05912ccf2a3c68f8__r1,7,2,0.09233
7,chatgpt__gb_05912ccf2a3c68f8__r1,8,2,0.01133
8,chatgpt__gb_05912ccf2a3c68f8__r1,9,2,0.01013
9,chatgpt__gb_0c0058305876a645__r1,1,1,0.80018



per-engine row counts:
engine
chatgpt        26354
claude        290279
gemini        375092
kimi          463909
perplexity     26158
dtype: int64


In [9]:
# --- Table 4 validation ---
expected_rows = int(v2['nli_evaluations'].sum())
print('rows == sum(nli_evaluations):', len(table4) == expected_rows, f'({len(table4)} vs {expected_rows})')
print('entail_score in [0, 1]      :',
      float(table4['entail_score'].min()) >= 0.0 and float(table4['entail_score'].max()) <= 1.0,
      f"(min={table4['entail_score'].min():.4f}, max={table4['entail_score'].max():.4f})")
print('every response_id in Table 1:', bool(table4['response_id'].isin(set(table1['response_id'])).all()))

# spot check: cross-reference one response's Table 4 rows against the raw sentence_detail
spot = table1[(table1.n_sources_fetched_ok > 0) & (table1.cleaned_n > 3)].iloc[0]['response_id']
v2['_rid'] = v2.engine + '__' + v2.query_id + '__r' + v2.run_index.astype(str)
sr = v2[v2['_rid'] == spot].iloc[0]
raw = sorted((int(e['idx']), int(s['pos']), round(float(s['score']), 6))
             for e in _L(sr.sentence_detail) for s in (_L(e.get('source_scores')) or []))
tab = sorted((int(a), int(b), round(float(c), 6))
             for a, b, c in table4[table4.response_id == spot][['sentence_idx', 'citation_position', 'entail_score']].values)
print(f'\nspot check {spot}: {len(tab)} rows, exact match vs raw sentence_detail:', raw == tab)
display(table4[table4.response_id == spot].head(8))

rows == sum(nli_evaluations): True (1181792 vs 1181792)
entail_score in [0, 1]      : True (min=0.0000, max=0.9998)
every response_id in Table 1: True

spot check chatgpt__gb_05912ccf2a3c68f8__r1: 9 rows, exact match vs raw sentence_detail: True


,response_id,sentence_idx,citation_position,entail_score
0,chatgpt__gb_05912ccf2a3c68f8__r1,1,2,0.03343
1,chatgpt__gb_05912ccf2a3c68f8__r1,2,2,0.03157
2,chatgpt__gb_05912ccf2a3c68f8__r1,3,2,0.53563
3,chatgpt__gb_05912ccf2a3c68f8__r1,4,2,0.45338
4,chatgpt__gb_05912ccf2a3c68f8__r1,5,2,0.35830
5,chatgpt__gb_05912ccf2a3c68f8__r1,6,2,0.78460
6,chatgpt__gb_05912ccf2a3c68f8__r1,7,2,0.09233
7,chatgpt__gb_05912ccf2a3c68f8__r1,8,2,0.01133


## Section 5 — Recomputing AIS and PAWC from Table 4

This section proves that Table 4, combined with the artefact flag and word count in Table 2, is
sufficient to reconstruct the stored metrics — no nested `sentence_detail` is consulted. Five responses
are drawn at random (one per engine) among those with at least one fetched source and more than three
surviving sentences.

For each, non-artefact sentences are re-indexed $k = 1 \dots N^{*}$ with linear position weight
$w^{*}_k = (N^{*} - k + 1)/N^{*}$. A sentence is supported when any of its `entail_score` values reaches
$\tau = 0.50$; AIS is the supported fraction. PAWC credits $wc \cdot w^{*}_k$ to every (sentence, source)
pair that reaches $\tau$, summed across pairs.

**Validation:** recomputed AIS matches `cleaned_ais` (within 1e-4) and recomputed PAWC matches
`cleaned_pawc_total` (within 1e-2) for all five responses.

In [10]:
TAU = 0.50
elig = table1[(table1.n_sources_fetched_ok > 0) & (table1.cleaned_n > 3)]
picks = elig.groupby('engine', group_keys=False).sample(n=1, random_state=42)

t2_by = {rid: sub for rid, sub in table2.groupby('response_id')}
t4_by = {rid: sub for rid, sub in table4.groupby('response_id')}

print(f"{'response_id':42s} {'AIS_rec':>8s} {'AIS_st':>8s} {'PAWC_rec':>10s} {'PAWC_st':>10s}  match")
all_ok = True
for _, prow in picks.iterrows():
    rid = prow['response_id']
    s2, s4 = t2_by[rid], t4_by[rid]
    # non-artefact sentences, re-indexed by ascending sentence_idx
    nonart = s2[~s2['is_artifact']].sort_values('sentence_idx')
    order = list(nonart['sentence_idx'])
    N_star = len(order)
    wc = dict(zip(nonart['sentence_idx'], nonart['word_count']))
    wstar = {idx: (N_star - k + 1) / N_star for k, idx in enumerate(order, start=1)}
    # supported (sentence, source) pairs at tau, restricted to non-artefact sentences
    sup_pairs = s4[s4['sentence_idx'].isin(set(order)) & (s4['entail_score'] >= TAU)]
    ais = sup_pairs['sentence_idx'].nunique() / N_star
    pawc = float(sum(wc[idx] * wstar[idx] for idx in sup_pairs['sentence_idx']))
    st_ais, st_pawc = prow['cleaned_ais'], prow['cleaned_pawc_total']
    ok = abs(ais - st_ais) < 1e-4 and abs(pawc - st_pawc) < 1e-2
    all_ok = all_ok and ok
    print(f'{rid:42s} {ais:8.4f} {st_ais:8.4f} {pawc:10.2f} {st_pawc:10.2f}  {ok}')

print('\nAll five responses reproduce stored AIS and PAWC from Table 4:', all_ok)

response_id                                 AIS_rec   AIS_st   PAWC_rec    PAWC_st  match
chatgpt__gb_a3a52437fd3f285b__r6             0.0000   0.0000       0.00       0.00  True
claude__gb_24d7696615085738__r2              0.5714   0.5714     487.94     487.94  True
gemini__gb_2c5ef36fc3e0c7bc__r8              0.9643   0.9643    1112.04    1112.04  True
kimi__gb_6c83b87033596338__r4                0.2000   0.2000       9.30       9.30  True
perplexity__gb_1843e132c89b2bb9__r1          0.7500   0.7500     241.25     241.25  True

All five responses reproduce stored AIS and PAWC from Table 4: True


## Section 6 — Persist the flat tables

The four tables are written to the gold layer and immediately reloaded to confirm their shapes survive
the round-trip. Table 4 is by far the largest (one row per NLI evaluation, ~1.18 million rows).

**Validation:** all four files saved; each reloads to the same shape; no input file modified.

In [11]:
OUT = {
    'data/scaleup/gold/flat_table1_responses.parquet':         table1,
    'data/scaleup/gold/flat_table2_sentences.parquet':         table2,
    'data/scaleup/gold/flat_table3_citations.parquet':         table3,
    'data/scaleup/gold/flat_table4_entailment_scores.parquet': table4,
}
for path, df in OUT.items():
    df.to_parquet(path, index=False)

print('saved tables (reloaded to verify shape):')
for path, df in OUT.items():
    mb = os.path.getsize(path) / 1e6
    reloaded = pd.read_parquet(path)
    print(f'  {os.path.basename(path):42s} {str(df.shape):14s} {mb:8.2f} MB  reload-matches={reloaded.shape == df.shape}')

saved tables (reloaded to verify shape):
  flat_table1_responses.parquet              (9992, 13)         0.27 MB  reload-matches=True
  flat_table2_sentences.parquet              (219897, 8)       19.51 MB  reload-matches=True
  flat_table3_citations.parquet              (58851, 5)         3.63 MB  reload-matches=True
  flat_table4_entailment_scores.parquet      (1181792, 4)       3.98 MB  reload-matches=True


### A note on the schema

- **Table 4 (entailment scores)** is the *atomic* table: each row is a single (sentence, source)
  entailment probability. Both AIS and PAWC are pure aggregations over it — AIS asks, per sentence,
  whether any score reaches the threshold; PAWC sums position-and-word-weighted credit over the scores
  that do. Section 5 demonstrates this reconstruction exactly.
- **Tables 1–3** supply the dimensional context: response-level metadata and stored metrics (Table 1),
  sentence text with its artefact flag and word count (Table 2), and the cited-source URLs with their
  fetch status (Table 3).
- Together they form a normalised relational schema joined on `response_id` (and, for the score and
  sentence tables, `sentence_idx`; for the score and citation tables, `citation_position`). An examiner
  or future researcher can therefore query the data with ordinary joins and group-bys, without parsing
  any nested array.

In [12]:
# Final summary
inputs_unmodified = all(_FP[p] == (os.path.getsize(p), round(os.path.getmtime(p), 3)) for p in INPUT_FILES)
print('Flat relational tables - summary')
print('  Table 1: %7d rows (responses)'         % len(table1))
print('  Table 2: %7d rows (sentences)'         % len(table2))
print('  Table 3: %7d rows (citations)'         % len(table3))
print('  Table 4: %7d rows (entailment scores)' % len(table4))
print('  All input files unmodified:', inputs_unmodified)
print('  Notebook: notebooks/scaleup/andre_flat_tables.ipynb')

Flat relational tables - summary
  Table 1:    9992 rows (responses)
  Table 2:  219897 rows (sentences)
  Table 3:   58851 rows (citations)
  Table 4: 1181792 rows (entailment scores)
  All input files unmodified: True
  Notebook: notebooks/scaleup/andre_flat_tables.ipynb
